# Share count dilution over time from XBRL

Track diluted shares outstanding by year or quarter from XBRL financials, to see dilution and buybacks next to cash flow.

Read it with charts at https://www.quantisnow.com/sec-api/guides/share-dilution-over-time. More notebooks: https://github.com/quantisnow/sec-edgar-notebooks. Set `QUANTISNOW_API_KEY` before running; make a key at https://www.quantisnow.com/sec-api/keys.

The financials endpoint carries `incomeStatement.sharesDiluted` on every period. A share count that climbs every year while revenue doesn't is dilution paying the bills; one that falls is a buyback doing its work.

## Diluted shares by year for one company

### How have Apple's revenue, free cash flow and diluted share count changed by year?

Annual records carry the diluted share count next to the cash flow, so dilution and buybacks show up year by year.

```shell
curl -X POST https://www.quantisnow.com/api/v1/financials \
  -H "Authorization: Bearer $QUANTISNOW_API_KEY" \
  -H "Content-Type: application/json" \
  -d '{
  "query": "ticker:AAPL AND fiscalPeriod:FY",
  "from": 0,
  "size": 5,
  "sort": [{"fiscalYear": {"order": "desc"}}]
}'
```

Response (real, first 2 of 15):

```json
{
  "total": {
    "value": 15,
    "relation": "eq"
  },
  "data": [
    {
      "id": "0000320193:2026:FY",
      "cik": "0000320193",
      "ticker": "AAPL",
      "companyName": "Apple Inc.",
      "fiscalYear": 2026,
      "fiscalPeriod": "FY",
      "periodStart": "2025-09-28",
      "periodEnd": "2026-09-26",
      "filedOn": "2026-11-03",
      "form": "10-K",
      "accessionNo": "0000320193-26-000026",
      "sector": "Technology",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 478834000000,
        "costOfRevenue": 245232000000,
        "grossProfit": 233602000000,
        "researchDevelopment": 45765000000,
        "sellingGeneralAdmin": 29774000000,
        "operatingIncome": 158063000000,
        "interestExpense": null,
        "pretaxIncome": 159237000000,
        "incomeTax": 27062000000,
        "netIncome": 132175000000,
        "epsBasic": 9.01,
        "epsDiluted": 8.98,
        "sharesDiluted": 14720306000
      },
      "balanceSheet": {
        "cash": 36305000000,
        "shortTermInvestments": 20417000000,
        "receivables": 43817000000,
        "inventory": 14541000000,
        "currentAssets": 173857000000,
        "goodwill": null,
        "totalAssets": 404548000000,
        "currentLiabilities": 169915000000,
        "shortTermDebt": 9906000000,
        "longTermDebt": 68208000000,
        "totalDebt": 78114000000,
        "totalLiabilities": 292455000000,
        "equity": 112093000000
      },
      "cashFlow": {
        "operatingCashFlow": 145097000000,
        "investingCashFlow": -17280000000,
        "financingCashFlow": -127446000000,
        "capex": 9041000000,
        "freeCashFlow": 136056000000,
        "dividendsPaid": 15721000000,
        "shareRepurchases": 86390000000,
        "stockCompensation": 13789000000,
        "depreciation": 13423000000
      }
    },
    {
      "id": "0000320193:2025:FY",
      "cik": "0000320193",
      "ticker": "AAPL",
      "companyName": "Apple Inc.",
      "fiscalYear": 2025,
      "fiscalPeriod": "FY",
      "periodStart": "2024-09-29",
      "periodEnd": "2025-09-27",
      "filedOn": "2026-11-03",
      "form": "10-K",
      "accessionNo": "0000320193-26-000026",
      "sector": "Technology",
      "currency": "USD",
      "derived": false,
      "incomeStatement": {
        "revenue": 416161000000,
        "costOfRevenue": 220960000000,
        "grossProfit": 195201000000,
        "researchDevelopment": 34550000000,
        "sellingGeneralAdmin": 27601000000,
        "operatingIncome": 133050000000,
        "interestExpense": null,
        "pretaxIncome": 132729000000,
        "incomeTax": 20719000000,
        "netIncome": 112010000000,
        "epsBasic": 7.49,
        "epsDiluted": 7.46,
        "sharesDiluted": 15004697000
      },
      "balanceSheet": {
        "cash": 35934000000,
        "shortTermInvestments": 18763000000,
        "receivables": 39777000000,
        "inventory": 5718000000,
        "currentAssets": 147957000000,
        "goodwill": null,
        "totalAssets": 359241000000,
        "currentLiabilities": 165631000000,
        "shortTermDebt": 12350000000,
        "longTermDebt": 78328000000,
        "totalDebt": 90678000000,
        "totalLiabilities": 285508000000,
        "equity": 73733000000
      },
      "cashFlow": {
        "operatingCashFlow": 111482000000,
        "investingCashFlow": 15195000000,
        "financingCashFlow": -120686000000,
        "capex": 12715000000,
        "freeCashFlow": 98767000000,
        "dividendsPaid": 15421000000,
        "shareRepurchases": 90711000000,
        "stockCompensation": 12863000000,
        "depreciation": 11698000000
      }
    }
  ]
}
```

## Year-over-year change in Python

In [ ]:
import os, requests

res = requests.post(
    "https://www.quantisnow.com/api/v1/financials",
    headers={"Authorization": f"Bearer {os.environ['QUANTISNOW_API_KEY']}"},
    json={
        "query": "ticker:AAPL AND fiscalPeriod:FY",
        "sort": "fiscalYear:desc",
        "size": 50,
    },
    timeout=30,
)
res.raise_for_status()
page = res.json()
print(page["total"]["value"], "matches")
rows = sorted(page["data"], key=lambda r: r["fiscalYear"])
for prev, cur in zip(rows, rows[1:]):
    a, b = prev["incomeStatement"]["sharesDiluted"], cur["incomeStatement"]["sharesDiluted"]
    if a and b:
        print(cur["fiscalYear"], f"{b / a - 1:+.1%}")

## From the command line

```shell
npx quantisnow financials AAPL --annual
```

Related: [ATM offerings](https://www.quantisnow.com/sec-api/guides/atm-offerings-dilution), [Buyback authorizations](https://www.quantisnow.com/sec-api/guides/buyback-authorizations), [Free cash flow screens](https://www.quantisnow.com/sec-api/guides/free-cash-flow-screen).